# 480 — Neurosynth meta-analytic regions

Assigns every electrode to **Neurosynth** association-test regions (the `neurosynth/`
folder: auditory · visual · motor control · phonological · semantic · lexical) by
sampling each FDR z-map at the electrode's fsaverage location (MNI305 → MNI152).
**Multi-label** — an electrode joins *every* region whose map is significant (z > `Z_THR`)
at its location. For each region it averages the member electrodes' concatenated
`[audio | picture | reading]` ERSP into a card. Exports to `outputs/pooling/neurosynth/`
(`neurosynth_labels.csv` + `neurosynth_info.json` + `region_cards/`), which the POOL web
page shows as a **'Neurosynth'** colour mode alongside Yeo.


In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


INPUT_DIR : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY | exists: True
OUTPUTS   : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\04_FBM_Pooling\outputs\pooling
COORDS    : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\coords | exists: True
conditions: ('audio', 'picture', 'reading') | zones: ('perception', 'pre_articulation', 'audio')
feature sets: ('hg', 'bands15') | window shapes: ('boxcar', 'gaussian')


In [2]:
# ---- knobs ----
GRID  = 'full'
Z_THR = 0.0            # 'inside' a region = its FDR z-map > this at the electrode (0 = any significant voxel)


## 1 — Assign electrodes to neurosynth regions, average, export


In [3]:
coords = P.load_coords()
out = P.export_neurosynth(INPUT_DIR, coords, P.OUTPUTS_ROOT / 'neurosynth', grid=GRID, z_thr=Z_THR)
print('neurosynth ->', out)


[lf_pool] neurosynth maps: ['auditory', 'lexical', 'motor control', 'phonological', 'semantic', 'visual']
[lf_pool] neurosynth: 5247 contacts, 1377 in >=1 region (z>0.0)
[lf_pool] neurosynth -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\04_FBM_Pooling\outputs\pooling\neurosynth  (6 region cards)
neurosynth -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\04_FBM_Pooling\outputs\pooling\neurosynth


## 2 — Membership summary


In [4]:
import json, pandas as pd
info = json.load(open(P.OUTPUTS_ROOT / 'neurosynth' / 'neurosynth_info.json'))
display(pd.DataFrame([{'region': k, 'n_contacts': v['n'], 'n_ersp': v.get('n_ersp', 0)} for k, v in info.items()]))
lab = pd.read_csv(P.OUTPUTS_ROOT / 'neurosynth' / 'neurosynth_labels.csv')
print('contacts in >=1 region:', int((lab['neurosynth_primary'].fillna('') != '').sum()), '/', len(lab))
lab.head()


,region,n_contacts,n_ersp
0,auditory,103,34
1,visual,311,148
2,motor control,85,36
3,phonological,536,205
4,lexical,81,31
5,semantic,793,281


contacts in >=1 region: 1377 / 5247


,patient,contact,neurosynth,neurosynth_primary
0,EL030,AL1,NaN,NaN
1,EL030,AL2,NaN,NaN
2,EL030,AL3,NaN,NaN
3,EL030,AL4,NaN,NaN
4,EL030,AL5,NaN,NaN
